In [ ]:
# install torchinfo library
! pip install torchinfo

In [ ]:
# install torchinfo library
! pip install accelerate -U

In [ ]:
# install transformers library
! pip install transformers

In [ ]:
# install datasets library
! pip install datasets

In [ ]:
import pandas as pd
import numpy as np
import itertools
import re
from torch.utils.data import Dataset
import torch
from torchinfo import summary
import string
import tensorflow as tf

In [ ]:
test_data = pd.read_csv(r'/content/test_data.csv')
train_data = pd.read_csv(r'/content/train_data.csv')
valid_data = pd.read_csv(r'/content/valid_data.csv')

In [ ]:
!pip install spacy

In [ ]:
!pip install emoji==1.4.1

In [ ]:
import emoji
import spacy

# load spaCy model
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser", "tagger"])

def preprocess_dataframe(df, text_column):

    def strip_emoji(text):
        return re.sub(emoji.get_emoji_regexp(), r"", text)

    def clean_hashtags(tweet):
        return re.sub(r'#(\w+)', r'\1', tweet)

    def clean_usernames(tweet):
        return re.sub(r'@(\w+)', '', tweet)

    def remove_urls(text):
        url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
        return re.sub(url_pattern, '', text)

    def filter_chars(text):
        return ' '.join(word for word in text.split() if '$' not in word and '&' not in word)

    def remove_mult_spaces(text):
        return re.sub(r"\s\s+", " ", text)

    def remove_numbers(text):
        return re.sub(r'\d+', '', text)

    # tokenization
    def preprocess_text(text):
        doc = nlp(text)
        tokens = [token.text.lower() for token in doc if not token.is_space]
        return ' '.join(tokens)

    df['text_clear'] = df[text_column].apply(strip_emoji)
    df['text_clear'] = df['text_clear'].apply(clean_hashtags)
    df['text_clear'] = df['text_clear'].apply(clean_usernames)
    df['text_clear'] = df['text_clear'].apply(remove_urls)

    df['text_clear'] = df['text_clear'].str.replace("_", " ", regex=True)
    df['text_clear'] = df['text_clear'].str.replace("-", " ", regex=True)
    df['text_clear'] = df['text_clear'].str.replace(r'\n\n-', "", regex=True)
    df['text_clear'] = df['text_clear'].str.replace(r'\n', "", regex=True)
    df['text_clear'] = df['text_clear'].str.replace(r"_[A-Za-z0-9]+", " ", regex=True)

    df['text_clear'] = df['text_clear'].replace(np.nan, '')
    df['text_clear'] = df['text_clear'].apply(filter_chars)
    df['text_clear'] = df['text_clear'].apply(remove_mult_spaces)
    df['text_clear'] = df['text_clear'].str.lower()
    df['text_clear'] = df['text_clear'].apply(remove_numbers)
    df['text_clear'] = df['text_clear'].apply(lambda x: re.sub(r'[^\w\s]', '', x))

    df['text_clear'] = df['text_clear'].apply(preprocess_text)
    df['text_clear'] = df['text_clear'].apply(remove_mult_spaces)

    return df

In [ ]:
for i in [train_data, valid_data, test_data]:
    preprocess_dataframe(i, 'tweet')

In [ ]:
# Load BERT Tokenizer from hugging Face
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("digitalepidemiologylab/covid-twitter-bert-v2")

In [ ]:
# convert data to list
train_texts = train_data['text_clear'].to_list()
train_labels = train_data['Stance_number'].to_list()
# convert data to list
val_texts = valid_data['text_clear'].to_list()
val_labels = valid_data['Stance_number'].to_list()

In [ ]:
# tokenize word with max lenght 512
train_encodings = tokenizer(train_texts, truncation=True,max_length=512)
val_encodings  = tokenizer(val_texts, truncation=True,max_length=512)

In [ ]:
# DataLoader Class
class DataLoader(Dataset):
    """
    Custom Dataset class for handling tokenized text data and corresponding labels.
    Inherits from torch.utils.data.Dataset.
    """
    def __init__(self, encodings, labels):
        """
        Initializes the DataLoader class with encodings and labels.

        Args:
            encodings (dict): A dictionary containing tokenized input text data
                              (e.g., 'input_ids', 'token_type_ids', 'attention_mask').
            labels (list): A list of integer labels for the input text data.
        """
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        """
        Returns a dictionary containing tokenized data and the corresponding label for a given index.

        Args:
            idx (int): The index of the data item to retrieve.

        Returns:
            item (dict): A dictionary containing the tokenized data and the corresponding label.
        """
        # Retrieve tokenized data for the given index
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        # Add the label for the given index to the item dictionary
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        """
        Returns the number of data items in the dataset.

        Returns:
            (int): The number of data items in the dataset.
        """
        return len(self.labels)


In [ ]:
# instantiate DataLoader class
train_dataloader = DataLoader(train_encodings, train_labels)
eval_dataloader = DataLoader(val_encodings, val_labels)

In [ ]:
from transformers import  AutoModelForSequenceClassification,TrainingArguments,Trainer

In [ ]:
# Load BERT Model from hugging Face
model = AutoModelForSequenceClassification.from_pretrained("digitalepidemiologylab/covid-twitter-bert-v2",num_labels=3)

In [ ]:
# use GPU if available
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model.to(device)

In [ ]:
summary(model) # see model structure

In [ ]:
# training confing
training_args = TrainingArguments(
    output_dir = 'training_dir',
    run_name="bert-finetune-run1",
    eval_strategy = 'epoch',
    save_strategy='no',
    num_train_epochs = 3,
    per_device_train_batch_size = 64,
    per_device_eval_batch_size = 16,
    learning_rate = 0.00005,
    logging_steps=10,          
    logging_dir="./logs",      
    report_to="none"           
)

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
def compute_metrics(pred):
    """
    Computes accuracy, F1, precision, and recall for a given set of predictions.

    Args:
        pred (obj): An object containing label_ids and predictions attributes.
            - label_ids (array-like): A 1D array of true class labels.
            - predictions (array-like): A 2D array where each row represents
              an observation, and each column represents the probability of
              that observation belonging to a certain class.

    Returns:
        dict: A dictionary containing the following metrics:
            - Accuracy (float): The proportion of correctly classified instances.
            - F1 (float): The macro F1 score, which is the harmonic mean of precision
              and recall. Macro averaging calculates the metric independently for
              each class and then takes the average.
            - Precision (float): The macro precision, which is the number of true
              positives divided by the sum of true positives and false positives.
            - Recall (float): The macro recall, which is the number of true positives
              divided by the sum of true positives and false negatives.
    """
    # Extract true labels from the input object
    labels = pred.label_ids

    # Obtain predicted class labels by finding the column index with the maximum probability
    preds = pred.predictions.argmax(-1)

    # Compute macro precision, recall, and F1 score using sklearn's precision_recall_fscore_support function
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')

    # Calculate the accuracy score using sklearn's accuracy_score function
    acc = accuracy_score(labels, preds)

    # Return the computed metrics as a dictionary
    return {
        'Accuracy': acc,
        'F1': f1,
        'Precision': precision,
        'Recall': recall
    }


In [ ]:
# create trainer class
trainer  = Trainer(
    model,
    training_args,
    train_dataset = train_dataloader,
    eval_dataset = eval_dataloader,
    tokenizer = tokenizer ,
    compute_metrics = compute_metrics
)

In [ ]:
trainer.train() # start train

In [ ]:
trainer.save_model(r'mymodel') # save model in drive

In [ ]:
from transformers import pipeline

In [ ]:
# # use model for inference
model = pipeline('text-classification',model='/content/mymodel',device=0)

In [ ]:
test_text = test_data['text_clear'].to_list()

In [ ]:
p_test = model(test_text)

In [ ]:
p_tests = []
for i in p_test:
  if '0' in i['label']:
    p_tests.append(0)
  if '1' in i['label']:
    p_tests.append(1)
  if '2' in i['label']:
    p_tests.append(2)


In [ ]:
type(p_tests)

In [ ]:
from sklearn.metrics import confusion_matrix,classification_report

In [ ]:
test_label = test_data['Stance_number'].to_list()

In [ ]:
confusion_matrix(p_tests ,test_label , labels = [0,1,2])

In [ ]:
print(classification_report(p_tests ,test_label))

Labeling our dataset

In [ ]:
df=pd.read_csv(r'/content/CCTD-2022_2024.csv')

In [ ]:
df_processed = preprocess_dataframe(df, 'tweet_text')

In [ ]:
def truncate_texts_to_length(df, column, max_length):
    # Truncate texts that are longer than max_length
    df[column] = df[column].apply(lambda x: x[:max_length] if len(x) > max_length else x)
    return df

df = truncate_texts_to_length(df, 'text_clear', 512)

In [ ]:
test_text= df['text_clear'].to_list()

In [ ]:
p_test = model(test_text) 

In [ ]:
p_test

In [ ]:
label_values = [int(d['label'].split('_')[1]) for d in p_test]

print(label_values)

In [ ]:
df=pd.DataFrame(label_values,columns=['stance'])